In [ ]:
# ================================
# INSTALL REQUIRED LIBRARIES
# ================================
# Downgrade datasets to avoid script error + install required libs
!pip uninstall -y datasets -q
!pip install datasets==2.19.0 transformers seqeval -q


# ================================
# IMPORT LIBRARIES
# ================================
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)


# ================================
# LOAD DATASET (CoNLL-2003)
# ================================
# This dataset contains tokens + POS/NER tags
dataset = load_dataset("conll2003")


# ================================
# LOAD MODEL & TOKENIZER
# ================================
# Using DistilBERT (lighter + faster)
model_name = "distilbert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model with correct number of labels
num_labels = len(dataset["train"].features["ner_tags"].feature.names)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)


# ================================
# TOKENIZATION + LABEL ALIGNMENT
# ================================
# Convert words into tokens and align labels properly
def tokenize_and_align_labels(examples):

    # Tokenize input words
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        is_split_into_words=True
    )

    labels = []

    # Align labels with tokens
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                # Ignore special tokens
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Assign correct label
                label_ids.append(label[word_idx])
            else:
                # Ignore repeated subword tokens
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    # Add labels to tokenized input
    tokenized_inputs["labels"] = labels
    return tokenized_inputs


# Apply tokenization to dataset
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


# ================================
# DATA COLLATOR
# ================================
# Helps in batching and padding properly
data_collator = DataCollatorForTokenClassification(tokenizer)


# ================================
# TRAINING CONFIGURATION
# ================================
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    logging_steps=50,
)


# ================================
# TRAINER SETUP
# ================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)


# ================================
# TRAIN MODEL
# ================================
trainer.train()


# ================================
# EVALUATE MODEL
# ================================
trainer.evaluate()


# ================================
# SAVE MODEL
# ================================
model.save_pretrained("./model")
tokenizer.save_pretrained("./model")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 8.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
